In [12]:
import pandas as pd
import numpy as np

ratings = pd.read_csv('/Users/ivan/Desktop/recommender-system/data/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

print("Размер данных:", ratings.shape)

print("\nКоличество пользователей:", ratings['user_id'].nunique())
print("Количество фильмов:", ratings['item_id'].nunique())
print("\nРаспределение оценок:")
print(ratings['rating'].value_counts().sort_index())

Размер данных: (100000, 4)

Количество пользователей: 943
Количество фильмов: 1682

Распределение оценок:
rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [8]:
!pip install scikit-surprise
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split

# Приводим данные в формат surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['user_id', 'item_id', 'rating']], reader)

# Разделяем
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("Количество оценок в train:", trainset.n_ratings)
print("Количество оценок в test:", len(testset))

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached scipy-1.18.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (62 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.18.0-cp312-cp312-macosx_12_0_arm64.whl (28.7 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Количество оценок в train: 80000
Количество оценок в test: 20000


In [13]:
from surprise import KNNBasic, accuracy

# Item-based KNN
sim_options = {
    'name': 'cosine',
    'user_based': False   # False = Item-based
}

model_knn = KNNBasic(k=40, sim_options=sim_options, random_state=42)
model_knn.fit(trainset)

# Предсказания на тесте
predictions_knn = model_knn.test(testset)

# Метрики
rmse_knn = accuracy.rmse(predictions_knn)
mae_knn = accuracy.mae(predictions_knn)

print(f"\nItem-Based KNN")
print(f"RMSE: {rmse_knn:.4f}")
print(f"MAE:  {mae_knn:.4f}")

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0264
MAE:  0.8104

Item-Based KNN
RMSE: 1.0264
MAE:  0.8104


In [14]:
from surprise import SVD

# SVD
model_svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model_svd.fit(trainset)

# Предсказания
predictions_svd = model_svd.test(testset)

# Метрики
rmse_svd = accuracy.rmse(predictions_svd)
mae_svd = accuracy.mae(predictions_svd)

print(f"\nSVD (Matrix Factorization)")
print(f"RMSE: {rmse_svd:.4f}")
print(f"MAE:  {mae_svd:.4f}")

RMSE: 0.9352
MAE:  0.7375

SVD (Matrix Factorization)
RMSE: 0.9352
MAE:  0.7375


In [15]:
from collections import defaultdict
import numpy as np

def precision_recall_ndcg_at_k(predictions, k=10, threshold=3.5):
    """
    Считает Precision@K, Recall@K и NDCG@K
    """
    # Группируем предсказания по пользователям
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = []
    recalls = []
    ndcgs = []

    for uid, user_ratings in user_est_true.items():
        # Сортируем по предсказанной оценке (убывание)
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Топ-K
        top_k = user_ratings[:k]

        # Релевантные фильмы (true rating >= threshold)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rel_and_rec = sum((true_r >= threshold) for (_, true_r) in top_k)

        # Precision@K
        precisions.append(n_rel_and_rec / k)

        # Recall@K
        recalls.append(n_rel_and_rec / n_rel if n_rel > 0 else 0)

        # NDCG@K
        dcg = 0.0
        for i, (_, true_r) in enumerate(top_k):
            if true_r >= threshold:
                dcg += 1.0 / np.log2(i + 2)

        # Ideal DCG
        ideal_ratings = sorted([true_r for (_, true_r) in user_ratings if true_r >= threshold], reverse=True)[:k]
        idcg = sum(1.0 / np.log2(i + 2) for i in range(len(ideal_ratings)))

        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

# Считаем для обеих моделей
prec_knn, rec_knn, ndcg_knn = precision_recall_ndcg_at_k(predictions_knn, k=10)
prec_svd, rec_svd, ndcg_svd = precision_recall_ndcg_at_k(predictions_svd, k=10)

print("Item-Based KNN (K=10):")
print(f"  Precision@10: {prec_knn:.4f}")
print(f"  Recall@10:    {rec_knn:.4f}")
print(f"  NDCG@10:      {ndcg_knn:.4f}")

print("\nSVD (K=10):")
print(f"  Precision@10: {prec_svd:.4f}")
print(f"  Recall@10:    {rec_svd:.4f}")
print(f"  NDCG@10:      {ndcg_svd:.4f}")

Item-Based KNN (K=10):
  Precision@10: 0.5594
  Recall@10:    0.7047
  NDCG@10:      0.7810

SVD (K=10):
  Precision@10: 0.5837
  Recall@10:    0.7214
  NDCG@10:      0.8289
